# Phase 3.2: Session 2 (Part B) — Active-Ledger Full System

In [ ]:
# Cell 1: Install Dependencies
print('Installing dependencies...')
!pip install -q diffusers accelerate wfdb flwr web3 py-solc-x pyyaml scikit-learn
from solcx import install_solc
install_solc('0.8.19')
!npm install -g ganache 2>/dev/null && echo '✅ Ganache installed' || echo '⚠️ Ganache install issue'
print('\n\u2705 All dependencies ready!')

In [ ]:
# Cell 2: Copy Repository Code to Working Directory
import shutil, os, glob

DATASET_PATHS = glob.glob('/kaggle/input/*')
print(f'Available datasets: {DATASET_PATHS}')
if not DATASET_PATHS:
    raise FileNotFoundError('No dataset found! Click "Add Data" → upload your zipped FYP-Blockchain-FL folder.')

repo_root = None
for base in DATASET_PATHS:
    if os.path.isfile(os.path.join(base, 'main.py')):
        repo_root = base
        break
    for item in os.listdir(base):
        candidate = os.path.join(base, item)
        if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, 'main.py')):
            repo_root = candidate
            break
    if repo_root:
        break
if repo_root is None:
    for base in DATASET_PATHS:
        for root, dirs, files in os.walk(base):
            if root.count(os.sep) - base.count(os.sep) > 4: break
            if 'main.py' in files and 'core' in dirs:
                repo_root = root
                break
        if repo_root: break
if repo_root is None:
    raise FileNotFoundError(f'Cannot locate main.py in your dataset.\nDataset contents: {[os.listdir(p) for p in DATASET_PATHS]}')

print(f'Repo root found: {repo_root}')
WORK = '/kaggle/working'
for item in os.listdir(repo_root):
    if item.startswith('.'): continue
    src, dst = os.path.join(repo_root, item), os.path.join(WORK, item)
    if os.path.isdir(src): shutil.copytree(src, dst, dirs_exist_ok=True)
    else: shutil.copy2(src, dst)

os.chdir(WORK)
print(f'\u2705 Code copied. Working dir: {os.getcwd()}')

In [ ]:
# Cell 3: Download + Preprocess + Partition Data
print('=' * 60)
print('STEP 1/3: Downloading MIT-BIH Arrhythmia records...')
print('=' * 60)
!rm -rf data/* checkpoints/*
!PYTHONPATH=/kaggle/working python core/download_data.py

print('\n' + '=' * 60)
print('STEP 2/3: Preprocessing ECG signals...')
print('=' * 60)
!PYTHONPATH=/kaggle/working python core/preprocess_data.py

print('\n' + '=' * 60)
print('STEP 3/3: Partitioning across 10 clients...')
print('=' * 60)
!PYTHONPATH=/kaggle/working python core/partition_data.py

import os
pdir = 'data/partitioned'
if os.path.exists(pdir):
    print(f'\n\u2705 Data ready — {len(os.listdir(pdir))} client partition directories found')
else:
    print('\u274c data/partitioned not found — check errors above')

In [ ]:
# Cell 4: Start Ganache & Deploy Smart Contracts
import subprocess, time
print('Starting Ganache for Active-Ledger...')
ganache_proc = subprocess.Popen(
    'NODE_OPTIONS=--max-old-space-size=8192 ganache -p 8545 --accounts 15 --deterministic --gasLimit 30000000',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(10)
if ganache_proc.poll() is not None:
    ganache_proc = subprocess.Popen(
        'NODE_OPTIONS=--max-old-space-size=8192 ganache-cli -p 8545 --accounts 15 --deterministic --gasLimit 30000000',
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(10)

if ganache_proc.poll() is None:
    print(f'✅ Ganache running (PID: {ganache_proc.pid})')
else:
    print('⚠️ Ganache failed to start.')

print('\nDeploying FLLogger smart contract...')
!PYTHONPATH=/kaggle/working python core/deploy_contract.py
print('\n✅ Blockchain infrastructure ready!')

In [ ]:
# Cell 5: RUN SESSION 2 — ACTIVE-LEDGER (PoC + DIFFUSION)
!PYTHONPATH=/kaggle/working python benchmarks/run_session2_active_ledger.py